# Evaluate CLACELL Package

In [1]:
# Scumi annotated labels

import anndata as ad
import scanpy as sc
# For saving results on HPC Cluster
import joblib
import pandas as pd
import os

# Load training data
#adata = ad.io.read_h5ad('../data/CellTypistDataset/global_annotated.h5ad')
adata = ad.io.read_h5ad('../data/CellTypistDataset/CountAdded_PIP_global_object_for_cellxgene_annotated.h5ad')

# Filter blood data
adata = adata[adata.obs['Organ'] == 'BLD'].copy()
print(adata)

# Use raw data instead of already preprocessed data
adata.X = adata.layers['counts'].copy()

# Preprocessing

# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

# Remove mitochondrial, ribosomal and hemoglobin
adata = adata[:, ~adata.var["mt"]].copy()
adata = adata[:, ~adata.var["ribo"]].copy()
adata = adata[:, ~adata.var["hb"]].copy()

# Doublet Detection
sc.pp.scrublet(adata, batch_key="Donor")
adata = adata[adata.obs['predicted_doublet'] == False].copy()


# Normalization

# Saving count data
adata.layers["counts"] = adata.X.copy()

# Normalizing to median total counts
sc.pp.normalize_total(adata, target_sum=1e4)
# Logarithmize the data
sc.pp.log1p(adata)

# Filtering Highly variable genes
print('Before filtering highly variable genes ---')
print(adata)

sc.pp.highly_variable_genes(adata, n_top_genes=10000)

# Apply filter
adata = adata[:, adata.var['highly_variable']].copy()

print('After filtering highly variable genes ---')
print(adata)

# Create train test split

# All Donors: ['621B', '637C', 'A35', 'A36', 'D496', 'D503']
donor_train = ['637C', 'A35', 'A36', 'D503']
donor_test = ['621B', 'D496']

adata_train = adata[
    adata.obs["Donor"].isin(donor_train)
].copy()

adata_test = adata[
    adata.obs["Donor"].isin(donor_test)
].copy()

# Check split
print(adata_train.obs['Donor'].unique())
print(adata_test.obs['Donor'].unique())

# Prepare Data for training
X_train = adata_train.to_df()
gene_names_train = adata_train.var_names
y_train = adata_train.obs['scumi-annotation']

X_test = adata_test.to_df()
gene_names_test = adata_test.var_names
y_test = adata_test.obs['scumi-annotation']

AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'Age_range_colors', 'Sex_colors', 'scrublet'
    obsm: 'X_umap'
    layers: 'counts'
Before filtering highly variable genes 

In [2]:
from sklearn.metrics import f1_score
import sys

#project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
#if project_path not in sys.path:
#    sys.path.append(project_path)

from clacell import CellClassifier

classifier = CellClassifier()

# 1. GridSearch
print("\n1. grid_search")
classifier.grid_search(X_train, y_train, X_test, y_test, n_jobs=3)

# 2. Train
print("\n2. train")
classifier.train(X_train, y_train, C=0.001)

# 3. Evaluate
print("\n3. evaluate")
classifier.evaluate(X_test, y_test)

# 4. Predict
print("\n4. predict")
predictions = classifier.predict(X_test)
print(f"Macro F1: {f1_score(y_test, predictions, average="macro")}")


1. grid_search
Fitting 3 folds for each of 50 candidates, totalling 150 fits


/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: Converg

[CV 1/3; 1/50] START C=1.229001659969549, class_weight=balanced, dual=False, penalty=l1, tol=0.0016310325248245904
[CV 1/3; 1/50] END C=1.229001659969549, class_weight=balanced, dual=False, penalty=l1, tol=0.0016310325248245904;, score=0.677 total time=   4.9s
[CV 1/3; 2/50] START C=0.06447310986105083, class_weight=balanced, dual=True, penalty=l2, tol=0.0007181399624574901
[CV 1/3; 2/50] END C=0.06447310986105083, class_weight=balanced, dual=True, penalty=l2, tol=0.0007181399624574901;, score=0.686 total time=   0.6s
[CV 2/3; 2/50] START C=0.06447310986105083, class_weight=balanced, dual=True, penalty=l2, tol=0.0007181399624574901
[CV 2/3; 2/50] END C=0.06447310986105083, class_weight=balanced, dual=True, penalty=l2, tol=0.0007181399624574901;, score=0.993 total time=   0.6s
[CV 3/3; 2/50] START C=0.06447310986105083, class_weight=balanced, dual=True, penalty=l2, tol=0.0007181399624574901
[CV 3/3; 2/50] END C=0.06447310986105083, class_weight=balanced, dual=True, penalty=l2, tol=0.000

/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 3/3; 1/50] START C=1.229001659969549, class_weight=balanced, dual=False, penalty=l1, tol=0.0016310325248245904
[CV 3/3; 1/50] END C=1.229001659969549, class_weight=balanced, dual=False, penalty=l1, tol=0.0016310325248245904;, score=0.963 total time=   7.2s
[CV 2/3; 3/50] START C=1.148886048199776, class_weight=None, dual=False, penalty=l1, tol=0.004289389323238861
[CV 2/3; 3/50] END C=1.148886048199776, class_weight=None, dual=False, penalty=l1, tol=0.004289389323238861;, score=0.992 total time=   4.8s
[CV 3/3; 4/50] START C=0.5027030357270226, class_weight=balanced, dual=False, penalty=l2, tol=0.00037221600518801866
[CV 3/3; 4/50] END C=0.5027030357270226, class_weight=balanced, dual=False, penalty=l2, tol=0.00037221600518801866;, score=0.969 total time=   2.8s
[CV 3/3; 6/50] START C=0.05785471715863724, class_weight=balanced, dual=True, penalty=l2, tol=0.00037193643182852195
[CV 3/3; 6/50] END C=0.05785471715863724, class_weight=balanced, dual=True, penalty=l2, tol=0.000371936431

...

Best parameters found: {'C': np.float64(0.0013034448595693861), 'class_weight': None, 'dual': False, 'penalty': 'l1', 'tol': np.float64(0.0002999604666719403)}
Evaluate model on test data...
2026-07-12 23:53:59,287 - INFO - --- In distribution testset ---
2026-07-12 23:54:03,183 - INFO - Baseline accuracy score 0.9088
2026-07-12 23:54:03,239 - INFO - Classification Report:
                     precision    recall  f1-score   support

             B cell       0.90      0.93      0.91       120
     CD14+ monocyte       1.00      1.00      1.00      2575
        CD4+ T cell       0.88      0.99      0.93      3910
   Cytotoxic T cell       0.98      0.60      0.74      1824
     Dendritic cell       0.00      0.00      0.00         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.75      0.97      0.84       791
        Plasma cell       0.00      0.00      0.00        49

           accuracy                           0.91      9281
          ma

2026-07-12 23:55:34,283 - ERROR - No out-of-distribution dataset provided. Skipping out-of-distribution tests.



Start final training with best parameters on complete training data...
Train models with parameters: {'C': np.float64(0.0013034448595693861), 'class_weight': None, 'dual': False, 'penalty': 'l1', 'tol': np.float64(0.0002999604666719403)}

[CV 2/3; 36/50] END C=0.002998458066338403, class_weight=balanced, dual=True, penalty=l2, tol=0.0007077432838220483;, score=0.993 total time=   0.8s
[CV 1/3; 37/50] START C=0.0820359452442842, class_weight=balanced, dual=False, penalty=l2, tol=0.00030683074489588934
[CV 1/3; 37/50] END C=0.0820359452442842, class_weight=balanced, dual=False, penalty=l2, tol=0.00030683074489588934;, score=0.663 total time=  14.3s
[CV 3/3; 39/50] START C=0.002121887550872272, class_weight=balanced, dual=True, penalty=l2, tol=0.0004738818051986002
[CV 3/3; 39/50] END C=0.002121887550872272, class_weight=balanced, dual=True, penalty=l2, tol=0.0004738818051986002;, score=0.971 total time=   1.0s
[CV 2/3; 42/50] START C=0.002077068461093455, class_weight=None, dual=False, 

2026-07-13 00:01:00,693 - ERROR - No out-of-distribution dataset provided. Skipping out-of-distribution tests.



4. predict
Infer new data...
Macro F1: 0.9250994495813557


In [3]:
from clacell import preprocess_data
import anndata as ad

adata_ood = ad.io.read_h5ad('../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_30_000_samples_tmp_test.h5ad')
adata_ood_processed = preprocess_data(adata_ood)

In [4]:
import pickle
# 3. Evaluate
print("\n3. evaluate")
with open("master_feature_importance_interleaved_marker_genes.pkl", "rb") as f:
    feature_importance = pickle.load(f)

X_ood = adata_ood_processed.to_df()
y_ood = adata_ood_processed.obs['scumi-annotation']
classifier.evaluate(X_test, y_test, X_ood=X_ood, y_ood=y_ood, feature_importances=feature_importance)


3. evaluate
Evaluate model on test data...
2026-07-13 00:03:08,516 - INFO - --- In distribution testset ---
2026-07-13 00:03:12,363 - INFO - Baseline accuracy score 0.9374
2026-07-13 00:03:12,433 - INFO - Classification Report:
                     precision    recall  f1-score   support

             B cell       1.00      0.99      1.00       120
     CD14+ monocyte       1.00      1.00      1.00      2575
        CD4+ T cell       0.91      0.99      0.95      3910
   Cytotoxic T cell       0.92      0.76      0.83      1824
     Dendritic cell       1.00      0.60      0.75         5
      Megakaryocyte       1.00      1.00      1.00         7
Natural killer cell       0.90      0.84      0.87       791
        Plasma cell       1.00      1.00      1.00        49

           accuracy                           0.94      9281
          macro avg       0.97      0.90      0.93      9281
       weighted avg       0.94      0.94      0.94      9281

2026-07-13 00:04:22,922 - INFO - Ran

Distribution In-Distribution                                            \
Category            Baseline Classification Report                       
Sub-Category         Overall                B cell                       
Metric              Accuracy             precision    recall  f1-score   
0                   0.937399                   1.0  0.991667  0.995816   

Distribution                                                                \
Category                                                                     
Sub-Category         CD14+ monocyte                            CD4+ T cell   
Metric       support      precision    recall f1-score support   precision   
0              120.0       0.998448  0.999612  0.99903  2575.0      0.9112   

Distribution  ...   Out-of-Distribution                              \
Category      ... Classification Report                     Dropout   
Sub-Category  ...          weighted avg                      Random   
Metric        ...                recall  f1-score  support    score   
0             ...              0.877053  0.856102  28980.0  0.87021   

Distribution                                                           \
Category     Consistency                            Dropout             
Sub-Category Num Samples Inconsistent Predictions   FI_0.1%   FI_0.5%   
Metric             Count                    Count  Accuracy  Accuracy   
0                  28980                        0  0.875638  0.852622   

Distribution                      
Category                          
Sub-Category   FI_1.0%   FI_2.0%  
Metric        Accuracy  Accuracy  
0             0.845342  0.763596  

[1 rows x 98 columns]

In [4]:
import anndata as ad
# Load training data
#adata = ad.io.read_h5ad('../data/CellTypistDataset/global_annotated.h5ad')
adata_unprocessed = ad.io.read_h5ad('../data/CellTypistDataset/CountAdded_PIP_global_object_for_cellxgene_annotated.h5ad')

# Filter blood data
adata_unprocessed = adata_unprocessed[adata_unprocessed.obs['Organ'] == 'BLD'].copy()
print(adata_unprocessed)

# Use raw data instead of already preprocessed data
adata_unprocessed.X = adata_unprocessed.layers['counts'].copy()

AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'Age_range_colors', 'Sex_colors', 'scrublet'
    obsm: 'X_umap'
    layers: 'counts'


In [6]:
from sklearn.metrics import f1_score
import sys
import os

#project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
#if project_path not in sys.path:
#    sys.path.append(project_path)

from clacell import CellClassifier

classifier = CellClassifier()

# 1. GridSearch
print("\n1. grid_search")
classifier.grid_search(adata_unprocessed, labels='scumi-annotation', n_jobs=3)

# 2. Train
print("\n2. train")
classifier.train(adata_unprocessed, labels='scumi-annotation', C=0.001)

# 3. Evaluate
print("\n3. evaluate")
classifier.evaluate(adata_unprocessed, labels='scumi-annotation')

# 4. Predict
print("\n4. predict")
predictions = classifier.predict(adata_unprocessed)
print(f"Macro F1: {f1_score(adata.obs['scumi-annotation'], predictions, average="macro")}")


1. grid_search
Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters found: {'C': np.float64(0.010830117702343428), 'class_weight': None, 'l1_ratio': 0.0, 'max_iter': 1000, 'solver': 'saga', 'tol': np.float64(0.0267208394227517)}
Evaluate model on test data...
--- In distribution testset ---
Baseline accuracy score 0.9933

                     precision    recall  f1-score   support

             B cell       1.00      1.00      1.00        89
     CD14+ monocyte       1.00      1.00      1.00      1640
        CD4+ T cell       1.00      0.99      1.00      3377
   Cytotoxic T cell       0.97      0.99      0.98       670
     Dendritic cell       0.97      1.00      0.98        29
      Megakaryocyte       0.94      0.97      0.96        33
Natural killer cell       0.99      0.99      0.99      1028
        Plasma cell       1.00      0.96      0.98        28

           accuracy                           0.99      6894
          macro avg       0.98      0.99

In [7]:
# Test scumi annotation in clacell package
from sklearn.metrics import f1_score
import sys
import os
import json

#project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
#if project_path not in sys.path:
#    sys.path.append(project_path)

from clacell import MarkerAnnotator

with open('../scumi-dev/R/marker_gene/human_pbmc_marker.json', "r", encoding="utf-8") as file:
    marker_genes = json.load(file)

annotator = MarkerAnnotator(marker_genes)

adata_lim = adata[:3000].copy()
del adata_lim.obs['scumi-annotation']

print("Before annotation")
print(adata_lim)

adata_lim = annotator.annotate(adata_lim)

print("After annotation")
print(adata_lim)

Before annotation
AnnData object with n_obs × n_vars = 3000 × 10000
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'Age_range_colors', 'Sex_colors', 'scrublet', 'log1p', 'hvg'
   

/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


After annotation
AnnData object with n_obs × n_vars = 3000 × 10000
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'Age_range_colors', 'Sex_colors', 'scrublet',

In [1]:
from clacell import preprocess_data
import anndata as ad
from clacell import CellClassifier

classifier = CellClassifier()

adata2 = ad.io.read_h5ad('../data/CellTypistDataset/CountAdded_PIP_global_object_for_cellxgene_annotated.h5ad')

adata2_preprocessed = preprocess_data(adata2)

classifier.train(adata2_preprocessed, labels='scumi-annotation', C=0.001)

Train models with parameters: {'C': 0.001}


In [2]:
# Scumi annotated labels

import anndata as ad
import scanpy as sc
# For saving results on HPC Cluster
import joblib
import pandas as pd
import os

# Load training data
#adata = ad.io.read_h5ad('../data/CellTypistDataset/global_annotated.h5ad')
adata = ad.io.read_h5ad('../data/CellTypistDataset/CountAdded_PIP_global_object_for_cellxgene_annotated.h5ad')

# Filter blood data
adata = adata[adata.obs['Organ'] == 'BLD'].copy()
print(adata)

# Use raw data instead of already preprocessed data
adata.X = adata.layers['counts'].copy()

# Preprocessing

# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

# Remove mitochondrial, ribosomal and hemoglobin
adata = adata[:, ~adata.var["mt"]].copy()
adata = adata[:, ~adata.var["ribo"]].copy()
adata = adata[:, ~adata.var["hb"]].copy()

# Doublet Detection
sc.pp.scrublet(adata, batch_key="Donor")
adata = adata[adata.obs['predicted_doublet'] == False].copy()


# Normalization

# Saving count data
adata.layers["counts"] = adata.X.copy()

# Normalizing to median total counts
sc.pp.normalize_total(adata, target_sum=1e4)
# Logarithmize the data
sc.pp.log1p(adata)
print(adata)

# Create train test split

# All Donors: ['621B', '637C', 'A35', 'A36', 'D496', 'D503']
donor_train = ['637C', 'A35', 'A36', 'D503']
donor_test = ['621B', 'D496']

adata_train = adata[
    adata.obs["Donor"].isin(donor_train)
].copy()

adata_test = adata[
    adata.obs["Donor"].isin(donor_test)
].copy()

# Check split
print(adata_train.obs['Donor'].unique())
print(adata_test.obs['Donor'].unique())

# Prepare Data for training
X_train = adata_train.to_df()
gene_names_train = adata_train.var_names
y_train = adata_train.obs['scumi-annotation']

X_test = adata_test.to_df()
gene_names_test = adata_test.var_names
y_test = adata_test.obs['scumi-annotation']

AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'Age_range_colors', 'Sex_colors', 'scrublet'
    obsm: 'X_umap'
    layers: 'counts'
AnnData object with n_obs × n_vars = 27

In [3]:
from sklearn.metrics import f1_score
import sys
import os

#project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
#if project_path not in sys.path:
#    sys.path.append(project_path)


# 4. Predict
print("\n4. predict")
predictions = classifier.predict(X_test)
print(f"Macro F1: {f1_score(y_test, predictions, average="macro")}")


4. predict
Infer new data...
Macro F1: 0.9402453219390878
